In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("pokazaniya.csv", sep = ";")

In [3]:
df

,anon_id,data,pokazanie,paketov_za_sutki
0,6478032,2025-10-01,18.690000,1
1,6478032,2025-10-02,18.774000,1
2,6478032,2025-10-03,18.855000,1
3,6478032,2025-10-04,18.895000,1
4,6478032,2025-10-05,18.933000,1
...,...,...,...,...
693438,9096678,2026-09-21,46.943000,10
693439,9096678,2026-09-25,47.594364,9
693440,9096678,2026-09-26,47.734000,13
693441,9096678,2026-09-28,47.995000,1


In [4]:
nulls = df.isnull().sum()
print(nulls)
print('ВЫВОД:', 'пропусков нет, заполнять и удалять нечего' if nulls.sum() == 0
      else 'есть пропуски: заполнить или исключить строки до расчёта')

anon_id             0
data                0
pokazanie           0
paketov_za_sutki    0
dtype: int64
ВЫВОД: пропусков нет, заполнять и удалять нечего


In [5]:
print(df.dtypes)
print('ВЫВОД:', 'data хранится как строка, приводим к datetime (следующая ячейка)'
      if df['data'].dtype == object else 'data уже datetime, преобразование не нужно')

anon_id               int64
data                 object
pokazanie           float64
paketov_za_sutki      int64
dtype: object
ВЫВОД: data хранится как строка, приводим к datetime (следующая ячейка)


In [6]:
df['data'] = pd.to_datetime(df['data'])

In [7]:
print(df.dtypes)
ok = pd.api.types.is_datetime64_any_dtype(df['data'])
print('ВЫВОД:', 'типы корректны, можно сортировать и считать разности дат' if ok
      else 'data всё ещё не datetime: проверить формат дат')

anon_id                      int64
data                datetime64[ns]
pokazanie                  float64
paketov_za_sutki             int64
dtype: object
ВЫВОД: типы корректны, можно сортировать и считать разности дат


In [8]:
df_sorted = df.sort_values(by = ['anon_id', 'data'])

In [9]:
display(df_sorted)
ordered = df_sorted.groupby('anon_id')['data'].apply(lambda x: x.is_monotonic_increasing).all()
dups = df_sorted.duplicated(['anon_id', 'data']).sum()
print(f'счётчиков: {df_sorted["anon_id"].nunique()}, дублей (счётчик, дата): {dups}')
print('ВЫВОД:', 'порядок дат верный' if ordered else 'порядок нарушен, пересортировать',
      '; дублей дат нет, агрегировать не нужно' if dups == 0 else '; есть дубли дат: оставить последнюю запись')

,anon_id,data,pokazanie,paketov_za_sutki
440708,1001589,2025-10-01,280.760000,1
440709,1001589,2025-10-02,280.840000,1
440710,1001589,2025-10-03,280.864000,2
440711,1001589,2025-10-04,280.910000,1
440712,1001589,2025-10-05,281.130000,1
...,...,...,...,...
173986,9997690,2026-09-26,725.350000,19
173987,9997690,2026-09-27,726.116111,25
173988,9997690,2026-09-28,727.044353,25
173989,9997690,2026-09-29,727.723727,25


счётчиков: 3000, дублей (счётчик, дата): 0
ВЫВОД: порядок дат верный ; дублей дат нет, агрегировать не нужно


In [10]:
# Расход относительно предыдущей записи счётчика и число суток между ними
g = df_sorted.groupby('anon_id')
df_sorted['delta'] = g['pokazanie'].diff()
df_sorted['days'] = g['data'].diff().dt.days

# Убираем float-шум (~1e-12), настоящие отрицательные значения и NaN первой записи остаются
noise = df_sorted['delta'].abs() <= 1e-6
df_sorted.loc[noise, 'delta'] = 0

In [11]:
df_sorted['gap'] = df_sorted['days'] > 1
df_sorted['sutochnoe'] = df_sorted['delta'] / df_sorted['days']

result = df_sorted[['anon_id', 'data', 'pokazanie', 'days', 'gap', 'sutochnoe']]
display(result)
n_gap = df_sorted['gap'].sum()
print(f'записей после пропуска дней: {n_gap} ({n_gap / len(df_sorted):.1%}), '
      f'счётчиков с пропусками: {df_sorted.loc[df_sorted["gap"], "anon_id"].nunique()}')
print('ВЫВОД: расход за пропущенные дни делим на число суток (среднесуточное), '
      'иначе за один «день» получился бы завышенный расход и ложные всплески')

,anon_id,data,pokazanie,days,gap,sutochnoe
440708,1001589,2025-10-01,280.760000,NaN,False,NaN
440709,1001589,2025-10-02,280.840000,1.0,False,0.080000
440710,1001589,2025-10-03,280.864000,1.0,False,0.024000
440711,1001589,2025-10-04,280.910000,1.0,False,0.046000
440712,1001589,2025-10-05,281.130000,1.0,False,0.220000
...,...,...,...,...,...,...
173986,9997690,2026-09-26,725.350000,1.0,False,0.906500
173987,9997690,2026-09-27,726.116111,1.0,False,0.766111
173988,9997690,2026-09-28,727.044353,1.0,False,0.928242
173989,9997690,2026-09-29,727.723727,1.0,False,0.679374


записей после пропуска дней: 74029 (10.7%), счётчиков с пропусками: 2696
ВЫВОД: расход за пропущенные дни делим на число суток (среднесуточное), иначе за один «день» получился бы завышенный расход и ложные всплески


In [12]:
x = result['sutochnoe'].dropna()
count = ((x > 0) & (x < 1)).sum()
print(count)
print(f'ВЫВОД: {count / len(x):.1%} записей лежат в диапазоне 0..1 м³/сут — это норма; '
      f'пороги аномалий задаём заметно выше (5 и 10 м³/сут), а не от этого диапазона')

439904
ВЫВОД: 63.7% записей лежат в диапазоне 0..1 м³/сут — это норма; пороги аномалий задаём заметно выше (5 и 10 м³/сут), а не от этого диапазона


# Анализ аномалий (пункты 1–6)

In [33]:
import numpy as np

s = df_sorted['sutochnoe']
# «нормальный» день: расход известен, не отрицательный и не нереальный (< 10 м³/сут)
df_sorted['clean'] = s.notna() & (s >= 0) & (s < 10)

def longest_streak(mask):
    """Для каждого счётчика: самая длинная серия подряд идущих записей с mask=True, в сутках."""
    brk = (~mask).groupby(df_sorted['anon_id']).cumsum()          # номер серии: растёт при каждом False
    run = df_sorted['days'].where(mask, 0).groupby([df_sorted['anon_id'], brk]).sum()
    return run.groupby(level=0).max()

def show(mask, title):
    """Сколько записей и счётчиков попало под критерий."""
    rows = df_sorted[mask]
    print(f"{title}: записей {len(rows)}, счётчиков {rows['anon_id'].nunique()}")
    return rows

## 1. Отрицательный расход

In [35]:
neg = show(df_sorted['sutochnoe'] < 0, '1. Отрицательный расход')
neg = neg.assign(pred=neg['pokazanie'] - neg['delta'])           # показание до падения
neg['тип'] = np.select(
    [neg['delta'] > -1, neg['pokazanie'] < 0.1 * neg['pred']],
    ['мелкое (< 1 м³)', 'сброс почти к нулю'],
    default='крупный откат')
print(neg.groupby('тип')['anon_id'].agg(записей='size', счётчиков='nunique'))
display(neg.sort_values('delta').head(10)[['anon_id', 'data', 'pred', 'pokazanie', 'delta', 'тип']]
        .rename(columns={'pred': 'было', 'pokazanie': 'стало'}))

serious = neg[neg['тип'] != 'мелкое (< 1 м³)']
small_only = set(neg['anon_id']) - set(serious['anon_id'])
print(f'ВЫВОД: {len(small_only)} сч. только с мелкими откатами — округление/дребезг, на проверку не отправляем; '
      f'{serious["anon_id"].nunique()} сч. со сбросом или крупным откатом — проверить замену счётчика и сбой передачи. '
      f'Первые: {serious.sort_values("delta")["anon_id"].drop_duplicates().head(5).tolist()}')

1. Отрицательный расход: записей 269, счётчиков 102
                    записей  счётчиков
тип                                   
крупный откат           149         57
мелкое (< 1 м³)         101         58
сброс почти к нулю       19         11


,anon_id,data,было,стало,delta,тип
438368,2519659,2026-06-09,4.273565e+06,72.726000,-4.273492e+06,сброс почти к нулю
438370,2519659,2026-09-25,4.273565e+06,72.726000,-4.273492e+06,сброс почти к нулю
442978,2491162,2026-07-10,3.006868e+06,380.814286,-3.006487e+06,сброс почти к нулю
455297,8908023,2026-09-08,2.684368e+06,1.049375,-2.684367e+06,сброс почти к нулю
397635,2159987,2025-10-19,3.264180e+03,315.120000,-2.949060e+03,сброс почти к нулю
266444,7891133,2025-10-19,2.529970e+03,236.581000,-2.293389e+03,сброс почти к нулю
163197,8190741,2026-02-06,1.927200e+03,615.774000,-1.311426e+03,крупный откат
415976,1423304,2026-09-13,1.797099e+03,486.252750,-1.310846e+03,крупный откат
398281,7293232,2026-04-17,2.007900e+03,697.100000,-1.310800e+03,крупный откат
146302,5704914,2026-03-01,1.646050e+03,335.267000,-1.310783e+03,крупный откат


ВЫВОД: 42 сч. только с мелкими откатами — округление/дребезг, на проверку не отправляем; 60 сч. со сбросом или крупным откатом — проверить замену счётчика и сбой передачи. Первые: [2519659, 2491162, 8908023, 2159987, 7891133]


## 2. Всплески

In [15]:
ABS_SPIKE = 5      # м³/сут — абсолютный порог
REL_N = 10         # во сколько раз больше обычного для прибора
REL_MIN = 1        # относительный порог работает только выше 1 м³/сут

# «обычный» расход прибора — медиана его нормальных дней с потреблением
usual = df_sorted[df_sorted['clean'] & (s > 0)].groupby('anon_id')['sutochnoe'].median()
df_sorted['usual'] = df_sorted['anon_id'].map(usual)

spike_abs = (s > ABS_SPIKE)
spike_rel = (s > REL_N * df_sorted['usual']) & (s > REL_MIN)
a = show(spike_abs, f'2а. Абсолютный (> {ABS_SPIKE} м³/сут)')
r = show(spike_rel, f'2б. Относительный (> {REL_N}× обычного и > {REL_MIN} м³/сут)')
both = set(a['anon_id']) & set(r['anon_id'])
print('Счётчиков в обоих:', len(both))
display(r.sort_values('sutochnoe', ascending=False).head(10)[['anon_id', 'data', 'sutochnoe', 'usual']])

only_rel = set(r['anon_id']) - set(a['anon_id'])
print(f'ВЫВОД: {len(both)} сч. срабатывают по обоим порогам — проверять первыми (реальный прорыв или сбой); '
      f'{len(only_rel)} сч. только по относительному — второй очередью (необычное для себя, но не огромное). '
      f'Первые: {a.sort_values("sutochnoe", ascending=False)["anon_id"].drop_duplicates().head(5).tolist()}')

2а. Абсолютный (> 5 м³/сут): записей 795, счётчиков 95
2б. Относительный (> 10× обычного и > 1 м³/сут): записей 705, счётчиков 164
Счётчиков в обоих: 81


,anon_id,data,sutochnoe,usual
442977,2491162,2026-07-09,3.005504e+06,0.193000
455296,8908023,2026-09-07,2.684039e+06,0.092299
438369,2519659,2026-06-11,2.136746e+06,0.039500
438367,2519659,2026-06-05,3.561244e+04,0.039500
397633,2159987,2025-10-17,2.834770e+03,0.150000
266442,7891133,2025-10-17,2.124960e+03,0.190000
282480,8973572,2025-10-17,6.081360e+02,0.034000
393009,5359103,2026-07-05,5.303171e+02,0.125000
92917,2030325,2026-07-20,4.259160e+02,0.072500
457995,4964924,2026-09-20,4.010470e+02,0.120000


ВЫВОД: 81 сч. срабатывают по обоим порогам — проверять первыми (реальный прорыв или сбой); 83 сч. только по относительному — второй очередью (необычное для себя, но не огромное). Первые: [2491162, 8908023, 2519659, 2159987, 7891133]


## 3. Нереальные значения

In [16]:
UNREAL = 10        # м³/сут — для квартирного счётчика физически невозможно
unreal = show(s >= UNREAL, f'3. Нереальный расход (>= {UNREAL} м³/сут)')
print('десятки м³/сут (10..1000):', (unreal['sutochnoe'] < 1000).sum(), 'записей;',
      'тысячи и больше (>= 1000):', (unreal['sutochnoe'] >= 1000).sum(), 'записей,',
      unreal[unreal['sutochnoe'] >= 1000]['anon_id'].nunique(), 'счётчиков')
display(unreal.sort_values('sutochnoe', ascending=False).head(10)[['anon_id', 'data', 'days', 'sutochnoe']])

huge = unreal.loc[unreal['sutochnoe'] >= 1000, 'anon_id'].unique()
print(f'ВЫВОД: {len(huge)} сч. с расходом >= 1000 м³/сут — это ошибка данных: исключить записи из статистики '
      f'и проверить передачу ({list(huge)}); остальные {unreal["anon_id"].nunique() - len(huge)} сч. — '
      f'проверить прорыв трубы или сбой показаний. Нормальные дни считаем без этих записей (флаг clean)')

3. Нереальный расход (>= 10 м³/сут): записей 421, счётчиков 66
десятки м³/сут (10..1000): 415 записей; тысячи и больше (>= 1000): 6 записей, 5 счётчиков


,anon_id,data,days,sutochnoe
442977,2491162,2026-07-09,1.0,3.005504e+06
455296,8908023,2026-09-07,1.0,2.684039e+06
438369,2519659,2026-06-11,2.0,2.136746e+06
438367,2519659,2026-06-05,120.0,3.561244e+04
397633,2159987,2025-10-17,1.0,2.834770e+03
266442,7891133,2025-10-17,1.0,2.124960e+03
282480,8973572,2025-10-17,1.0,6.081360e+02
393009,5359103,2026-07-05,1.0,5.303171e+02
92917,2030325,2026-07-20,1.0,4.259160e+02
457995,4964924,2026-09-20,1.0,4.010470e+02


ВЫВОД: 5 сч. с расходом >= 1000 м³/сут — это ошибка данных: исключить записи из статистики и проверить передачу ([2159987, 2491162, 2519659, 7891133, 8908023]); остальные 61 сч. — проверить прорыв трубы или сбой показаний. Нормальные дни считаем без этих записей (флаг clean)


## 4. Утечка

In [17]:
LEAK_DAYS = 60     # суток подряд
LEAK_FLOOR = 0.2   # расход КАЖДЫЙ день не ниже 0.2 м³/сут (200 л): прибор «никогда не отдыхает»

streak_leak = longest_streak(s >= LEAK_FLOOR)
leak = streak_leak[streak_leak >= LEAK_DAYS]
print(f'4. Утечка (>= {LEAK_DAYS} сут подряд расход >= {LEAK_FLOOR} м³/сут): счётчиков {len(leak)}')
never_zero = df_sorted[s.notna()].groupby('anon_id')['sutochnoe'].apply(lambda x: (x > 0).all())
print('из них вообще ни одного нулевого дня за весь период:', never_zero.reindex(leak.index).sum())
# чувствительность к порогу
for fl in (0.1, 0.2, 0.5):
    print(f'  порог {fl} м³/сут:', (longest_streak(s >= fl) >= LEAK_DAYS).sum(), 'счётчиков')
display(leak.sort_values(ascending=False).head(10))

strong = (longest_streak(s >= 0.5) >= LEAK_DAYS)
strong = strong[strong].index
print(f'ВЫВОД: {len(strong)} сч. с серией >= {LEAK_DAYS} сут при расходе >= 0.5 м³/сут — выезд на утечку первыми '
      f'({list(strong[:5])}); остальные {len(leak) - len(set(strong) & set(leak.index))} сч. из {len(leak)} — '
      f'проверить удалённо (сантехника, бытовая техника, течь в бачке)')

4. Утечка (>= 60 сут подряд расход >= 0.2 м³/сут): счётчиков 226
из них вообще ни одного нулевого дня за весь период: 76
  порог 0.1 м³/сут: 505 счётчиков
  порог 0.2 м³/сут: 226 счётчиков
  порог 0.5 м³/сут: 49 счётчиков


anon_id
4945470    365.0
5754693    364.0
3788388    364.0
9277360    363.0
1648954    363.0
1606847    363.0
7657012    362.0
3239572    362.0
8975533    361.0
9719703    352.0
Name: days, dtype: float64

ВЫВОД: 49 сч. с серией >= 60 сут при расходе >= 0.5 м³/сут — выезд на утечку первыми ([1138644, 1401643, 1581018, 1606847, 1648954]); остальные 177 сч. из 226 — проверить удалённо (сантехника, бытовая техника, течь в бачке)


## 5. Нулевое потребление при живом приборе

In [18]:
ZERO_DAYS = 30     # суток подряд нулевого расхода

streak_zero = longest_streak(s == 0)
packets = df_sorted[s == 0].groupby('anon_id')['paketov_za_sutki'].mean()   # пакетов/сут в нулевые дни
zero = streak_zero[streak_zero >= ZERO_DAYS]
print(f'5. Нулевое потребление (>= {ZERO_DAYS} сут подряд, пакеты идут): счётчиков {len(zero)}')
print('среднее число пакетов в сутки в эти дни —', packets.reindex(zero.index).describe().round(1).to_dict())
always_zero = df_sorted[s.notna()].groupby('anon_id')['sutochnoe'].apply(lambda x: (x == 0).all())
n_always = int(always_zero.reindex(zero.index).sum())
print('из них ноль весь период наблюдения:', n_always)
display(zero.sort_values(ascending=False).head(10))

ids_always = always_zero.reindex(zero.index)
print(f'ВЫВОД: {n_always} сч. с нулём весь период — отправить на проверку первыми (застрявший или отключённый '
      f'счётчик при живой связи; {list(ids_always[ids_always].index[:5])}); остальные {len(zero) - n_always} сч. — '
      f'вероятно пустая квартира, сверить с абонентской базой и событиями. Пакеты есть всегда, '
      f'поэтому по ним причину не определить')

5. Нулевое потребление (>= 30 сут подряд, пакеты идут): счётчиков 1133
среднее число пакетов в сутки в эти дни — {'count': 1133.0, 'mean': 6.0, 'std': 6.9, 'min': 1.0, '25%': 1.3, '50%': 3.2, '75%': 7.9, 'max': 47.5}
из них ноль весь период наблюдения: 166


anon_id
4137673    365.0
9182478    365.0
1268324    365.0
1223313    365.0
1831470    365.0
6119963    365.0
5661629    365.0
2351697    365.0
2204880    365.0
6714990    365.0
Name: days, dtype: float64

ВЫВОД: 166 сч. с нулём весь период — отправить на проверку первыми (застрявший или отключённый счётчик при живой связи; [1033804, 1133197, 1215340, 1223313, 1263288]); остальные 967 сч. — вероятно пустая квартира, сверить с абонентской базой и событиями. Пакеты есть всегда, поэтому по ним причину не определить


## 6. Ступенька

In [19]:
RATIO = 3          # вторая половина больше первой минимум в 3 раза
MIN_DAYS = 30      # минимум нормальных дней в каждой половине
MIN_DIFF = 0.1     # и разница не меньше 0.1 м³/сут
MIN_BASE = 0.01    # в 1-й половине расход не «почти ноль» (иначе отношение бессмысленно)

c = df_sorted[df_sorted['clean']].copy()
mid = c.groupby('anon_id')['data'].transform(lambda x: x.min() + (x.max() - x.min()) / 2)  # середина периода
c['половина'] = np.where(c['data'] <= mid, 1, 2)
half = c.groupby(['anon_id', 'половина'])['sutochnoe'].agg(['mean', 'median', 'size']).unstack()
h1 = half.xs(1, axis=1, level=1)
h2 = half.xs(2, axis=1, level=1)
step = (h1['size'] >= MIN_DAYS) & (h2['size'] >= MIN_DAYS) & (h1['mean'] >= MIN_BASE) \
     & (h2['mean'] >= RATIO * h1['mean']) & (h2['mean'] - h1['mean'] >= MIN_DIFF) \
     & (h2['median'] > h1['mean'])          # не из-за пары выбросов: медиана 2-й половины тоже выше
res = pd.DataFrame({'ср_1': h1['mean'], 'ср_2': h2['mean'], 'во сколько раз': h2['mean'] / h1['mean']})[step]
print(f'6. Ступенька (2-я половина >= {RATIO}× первой): счётчиков {len(res)}')
top = res.sort_values('во сколько раз', ascending=False)
display(top.head(10))

big = top[top['ср_2'] >= 1]
print(f'ВЫВОД: {len(big)} сч. с новым уровнем >= 1 м³/сут — проверить на скрытую утечку или самовольное подключение '
      f'({big.index[:5].tolist()}); остальные {len(res) - len(big)} сч. — вероятно смена жильцов или сезонность, '
      f'проверить во вторую очередь')

6. Ступенька (2-я половина >= 3× первой): счётчиков 49


,ср_1,ср_2,во сколько раз
anon_id,,,
3761383,0.034213,2.303932,67.341687
4673022,0.010048,0.248835,24.765854
9706252,0.011288,0.274946,24.356346
4050676,0.012678,0.224773,17.728759
9194829,0.018697,0.288410,15.425215
4565193,0.016493,0.251232,15.232946
1045488,0.019053,0.271153,14.231592
4104576,0.010913,0.154141,14.125162
2704749,0.040237,0.515625,12.814566


ВЫВОД: 3 сч. с новым уровнем >= 1 м³/сут — проверить на скрытую утечку или самовольное подключение ([3761383, 3743114, 1581018]); остальные 46 сч. — вероятно смена жильцов или сезонность, проверить во вторую очередь


In [20]:
# Сводка: по каждому счётчику — какие пункты 1–6 сработали
flags = pd.DataFrame(index=df_sorted['anon_id'].unique())
flags['1_отрицательный'] = flags.index.isin(neg['anon_id'])
flags['2_всплеск'] = flags.index.isin(pd.concat([a, r])['anon_id'])
flags['3_нереальный'] = flags.index.isin(unreal['anon_id'])
flags['4_утечка'] = flags.index.isin(leak.index)
flags['5_ноль'] = flags.index.isin(zero.index)
flags['6_ступенька'] = flags.index.isin(res.index)
flags['сработало_пунктов'] = flags.sum(axis=1)
print(flags.drop(columns='сработало_пунктов').sum())
display(flags.sort_values('сработало_пунктов', ascending=False).head(10))

print(flags['сработало_пунктов'].value_counts().sort_index().to_dict())
n3 = (flags['сработало_пунктов'] >= 3).sum()
n2 = (flags['сработало_пунктов'] == 2).sum()
print(f'ВЫВОД: {n3} сч. с тремя и более пунктами — на проверку первыми; {n2} сч. с двумя — второй очередью; '
      f'счётчики с одним пунктом — по результатам разбора пунктов выше')

1_отрицательный     102
2_всплеск           178
3_нереальный         66
4_утечка            226
5_ноль             1133
6_ступенька          49
dtype: int64


,1_отрицательный,2_всплеск,3_нереальный,4_утечка,5_ноль,6_ступенька,сработало_пунктов
8498147,True,True,True,False,True,True,5
1184793,True,True,True,False,True,False,4
9317608,True,True,True,True,False,False,4
7508199,True,True,True,True,False,False,4
2491162,True,True,True,False,True,False,4
5115252,True,True,True,False,True,False,4
7826125,True,True,True,False,True,False,4
2519659,True,True,True,True,False,False,4
7293232,True,True,True,False,True,False,4
8727125,False,True,True,False,True,True,4


{0: 1489, 1: 1353, 2: 90, 3: 52, 4: 15, 5: 1}
ВЫВОД: 68 сч. с тремя и более пунктами — на проверку первыми; 90 сч. с двумя — второй очередью; счётчики с одним пунктом — по результатам разбора пунктов выше


# 7. Связь аномалий с событиями «магнит»
### 7.1 Счётчики с магнитом и без

In [21]:
from scipy.stats import fisher_exact, wilcoxon

sobytiya = pd.read_csv("sobytiya.csv", sep=";")
sobytiya['data'] = pd.to_datetime(sobytiya['data'])
print('строк в файле:', len(sobytiya), '| уникальных:', len(sobytiya.drop_duplicates()))
sobytiya = sobytiya.drop_duplicates()          # в файле много полных дублей
print(sobytiya['sobytie'].value_counts())

# 7.1 Уровень счётчика: доля аномалий у счётчиков с магнитом и без
flags['магнит'] = flags.index.isin(sobytiya.loc[sobytiya['sobytie'] == 'магнит', 'anon_id'])
print('счётчиков с магнитом:', flags['магнит'].sum(), 'из', len(flags))
rows = []
for col in [c for c in flags.columns if c[0].isdigit()]:
    rows.append({'аномалия': col,
                 'доля с магнитом': flags.loc[flags['магнит'], col].mean(),
                 'доля без магнита': flags.loc[~flags['магнит'], col].mean(),
                 'p (Фишер)': fisher_exact(pd.crosstab(flags['магнит'], flags[col]))[1]})
tbl71 = pd.DataFrame(rows).round(4)
display(tbl71)

more = tbl71[(tbl71['p (Фишер)'] < 0.05) & (tbl71['доля с магнитом'] > tbl71['доля без магнита'])]['аномалия'].tolist()
less = tbl71[(tbl71['p (Фишер)'] < 0.05) & (tbl71['доля с магнитом'] < tbl71['доля без магнита'])]['аномалия'].tolist()
print('ВЫВОД:', f'магнит повышает долю аномалий {more}: делать его признаком для проверки' if more
      else f'магнит не повышает долю ни одной аномалии (у магнитных реже: {less}) — приоритетом для проверки его не делаем')

строк в файле: 34009 | уникальных: 10786
магнит    10164
сброс       353
Холл        269
Name: sobytie, dtype: int64
счётчиков с магнитом: 671 из 3000


,аномалия,доля с магнитом,доля без магнита,p (Фишер)
0,1_отрицательный,0.0253,0.0365,0.1837
1,2_всплеск,0.0432,0.0640,0.0510
2,3_нереальный,0.0060,0.0266,0.0005
3,4_утечка,0.0656,0.0781,0.3190
4,5_ноль,0.2697,0.4088,0.0000
5,6_ступенька,0.0179,0.0159,0.7299


ВЫВОД: магнит не повышает долю ни одной аномалии (у магнитных реже: ['3_нереальный', '5_ноль']) — приоритетом для проверки его не делаем


### 7.2 Меняется ли расход после события

In [22]:
# 7.2 Расход за 14 суток до и после события у того же счётчика
W = 14
normal = df_sorted.loc[df_sorted['clean'], ['anon_id', 'data', 'sutochnoe']]
by_meter = {k: v.set_index('data')['sutochnoe'] for k, v in normal.groupby('anon_id')}

def before_after(ev):
    before, after = [], []
    for aid, dt in zip(ev['anon_id'], ev['data']):
        x = by_meter.get(aid)
        if x is None:
            continue
        b = x[(x.index >= dt - pd.Timedelta(days=W)) & (x.index < dt)]
        a = x[(x.index > dt) & (x.index <= dt + pd.Timedelta(days=W))]
        if len(b) >= 5 and len(a) >= 5:          # достаточно дней по обе стороны
            before.append(b.mean())
            after.append(a.mean())
    return np.array(before), np.array(after)

out = []
for tip in ['магнит', 'сброс', 'Холл']:
    b, a = before_after(sobytiya[sobytiya['sobytie'] == tip])
    out.append({'событие': tip, 'событий': len(b), 'до': b.mean(), 'после': a.mean(),
                'доля со снижением': (a < b).mean(), 'p (Вилкоксон)': wilcoxon(b, a)[1]})
tbl72 = pd.DataFrame(out).round(4)
display(tbl72)

drop = tbl72[(tbl72['p (Вилкоксон)'] < 0.05) & (tbl72['после'] < tbl72['до'])]['событие'].tolist()
print('ВЫВОД:', f'после событий {drop} расход значимо падает: счётчики с ними проверять на остановку'
      if drop else 'расход после событий значимо не падает: останавливать счётчик магнитом по этим данным не получается, '
                   'отдельную проверку «после магнита» не вводим')

,событие,событий,до,после,доля со снижением,p (Вилкоксон)
0,магнит,5695,0.1606,0.1676,0.4033,0.0570
1,сброс,84,0.0567,0.0557,0.5357,0.6928
2,Холл,71,0.1421,0.1583,0.5070,0.4043


ВЫВОД: расход после событий значимо не падает: останавливать счётчик магнитом по этим данным не получается, отдельную проверку «после магнита» не вводим


### 7.3 Падение показаний рядом с событием

In [23]:
# 7.3 Отрицательный расход рядом с событием (±3 суток) против фона
DAYS_NEAR = 3
neg_rows = neg[['anon_id', 'data']]
all_rows = df_sorted.loc[df_sorted['sutochnoe'].notna(), ['anon_id', 'data']]

def near_event(rows, ev):
    m = rows.merge(ev, on='anon_id', suffixes=('', '_соб'))
    m = m[(m['data'] - m['data_соб']).abs() <= pd.Timedelta(days=DAYS_NEAR)]
    return m[['anon_id', 'data']].drop_duplicates()

out = []
for tip in ['магнит', 'сброс', 'Холл']:
    ev = sobytiya[sobytiya['sobytie'] == tip]
    n_neg, n_all = len(near_event(neg_rows, ev)), len(near_event(all_rows, ev))
    out.append({'событие': tip, 'отрицательных рядом': n_neg, 'записей рядом': n_all,
                'доля отриц. рядом с событием': n_neg / n_all,
                'доля отриц. в целом': len(neg_rows) / len(all_rows)})
tbl73 = pd.DataFrame(out).round(4)
display(tbl73)

tbl73['раз выше фона'] = tbl73['доля отриц. рядом с событием'] / tbl73['доля отриц. в целом']
linked = tbl73[(tbl73['раз выше фона'] >= 3) & (tbl73['отрицательных рядом'] >= 5)]['событие'].tolist()
print('ВЫВОД:', f'падения показаний связаны с событиями {linked}: отрицательные записи рядом с ними считать '
      f'реальным сбросом/заменой счётчика, а не сбоем данных; у остальных падений причину искать отдельно'
      if linked else 'связи падений с событиями нет, падения считаем сбоем данных')

,событие,отрицательных рядом,записей рядом,доля отриц. рядом с событием,доля отриц. в целом
0,магнит,5,12436,0.0004,0.0004
1,сброс,16,590,0.0271,0.0004
2,Холл,0,199,0.0000,0.0004


ВЫВОД: падения показаний связаны с событиями ['сброс']: отрицательные записи рядом с ними считать реальным сбросом/заменой счётчика, а не сбоем данных; у остальных падений причину искать отдельно


## Вывод по пункту 7

In [24]:
# Итог: список счётчиков на проверку и очередность
f = flags.copy()
serious_neg = set(neg.loc[neg['тип'] != 'мелкое (< 1 м³)', 'anon_id'])
f['серьёзный_отрицательный'] = f.index.isin(serious_neg)
f['ноль_весь_период'] = f.index.isin(always_zero[always_zero].index) & f['5_ноль']

# очередь 1: >= 3 пунктов, либо нереальный расход, либо ноль весь период (застрявший счётчик)
# очередь 2: >= 2 пунктов, либо серьёзное падение, либо утечка/ступенька
# очередь 3: любой другой сработавший пункт (кроме одних лишь мелких откатов)
only_small_neg = (f['1_отрицательный'] & ~f['серьёзный_отрицательный']
                  & (f['сработало_пунктов'] == 1))
f['очередь'] = np.select(
    [(f['сработало_пунктов'] >= 3) | f['3_нереальный'] | f['ноль_весь_период'],
     (f['сработало_пунктов'] >= 2) | f['серьёзный_отрицательный'] | f['4_утечка'] | f['6_ступенька'],
     (f['сработало_пунктов'] >= 1) & ~only_small_neg],
    [1, 2, 3], default=0)

queue = f[f['очередь'] > 0].sort_values(['очередь', 'сработало_пунктов'], ascending=[True, False])
queue.to_csv('check_list.csv', sep=';', encoding='utf-8-sig', index_label='anon_id')
print(f['очередь'].value_counts().sort_index().rename({0: '0 (не проверять)'}).to_dict())
display(queue.head(15))
q1, q2, q3 = [(f['очередь'] == i).sum() for i in (1, 2, 3)]
print(f'ВЫВОД: очередь 1 — {q1} сч. (выезд/проверка сразу), очередь 2 — {q2} сч. (удалённая проверка), '
      f'очередь 3 — {q3} сч. (наблюдение). Остальные {(f["очередь"] == 0).sum()} сч. без замечаний. '
      f'Магнит приоритетом не считаем (пункт 7). Список сохранён в check_list.csv')

{'0 (не проверять)': 1516, 1: 238, 2: 285, 3: 961}


,1_отрицательный,2_всплеск,3_нереальный,4_утечка,5_ноль,6_ступенька,сработало_пунктов,магнит,серьёзный_отрицательный,ноль_весь_период,очередь
8498147,True,True,True,False,True,True,5,False,True,False,1
1184793,True,True,True,False,True,False,4,False,True,False,1
2351232,True,True,True,False,True,False,4,False,True,False,1
2491162,True,True,True,False,True,False,4,False,True,False,1
2519659,True,True,True,True,False,False,4,False,True,False,1
2665800,True,True,True,True,False,False,4,False,True,False,1
4226606,True,True,True,False,True,False,4,False,True,False,1
4788930,False,True,False,True,True,True,4,False,False,False,1
5115252,True,True,True,False,True,False,4,False,True,False,1
5927111,True,True,True,True,False,False,4,False,True,False,1


ВЫВОД: очередь 1 — 238 сч. (выезд/проверка сразу), очередь 2 — 285 сч. (удалённая проверка), очередь 3 — 961 сч. (наблюдение). Остальные 1516 сч. без замечаний. Магнит приоритетом не считаем (пункт 7). Список сохранён в check_list.csv


# 8. Собственные находки

In [29]:
# 8.0 Таблица по счётчикам: паспорт прибора + показатели потребления
pribory = pd.read_csv("pribory.csv", sep=";")
pribory['vypusk'] = pd.to_datetime(pribory['mesyac_vypuska'] + '-01')

mm = df_sorted[df_sorted['clean']].groupby('anon_id').agg(
    ср_расход=('sutochnoe', 'mean'),
    доля_нулей=('sutochnoe', lambda x: (x == 0).mean()),
    пакетов=('paketov_za_sutki', 'mean'))
mm = pribory.set_index('anon_id').join(mm)
mm['ноль_весь_период'] = always_zero.reindex(mm.index)
mm['отрицательный'] = mm.index.isin(neg['anon_id'])
mm['нереальный'] = mm.index.isin(unreal['anon_id'])
mm['старый'] = mm['vypusk'] < '2020-01-01'          # выпуск до 2020 года

def frac(k, n):
    """Показывает расчёт доли: k/n = процент."""
    return f'{k}/{n} = {k / n:.1%}'

display(mm.head())
print('ВЫВОД: таблица готова, дальше сравниваем группы приборов по типу, возрасту, станции и пакетам')

,bs,tip,model,mesyac_vypuska,vypusk,ср_расход,доля_нулей,пакетов,ноль_весь_период,отрицательный,нереальный,старый
anon_id,,,,,,,,,,,,
6478032,BS-01,aqua2_stm,АКВА L110 D15 C E,2022-10,2022-10-01,0.177182,0.003175,5.050794,False,False,False,False
4743501,BS-01,aqua2_stm,АКВА L110 D15 B E,2025-10,2025-10-01,0.000000,1.000000,1.203448,True,False,False,False
4382302,BS-01,aqua2_nvt_wa_stm,АКВА L110 D15 B E,2025-11,2025-11-01,0.000006,0.978947,2.063158,False,False,False,False
9325487,BS-01,aqua2_nvt_wa_stm,АКВА L110 D15 C E,2023-09,2023-09-01,0.178314,0.140411,6.085616,False,False,False,False
6754005,BS-01,aqua2_nvt_wa_stm,АКВА L110 D15 C E,2025-12,2025-12-01,0.005260,0.881188,3.821782,False,False,False,False


ВЫВОД: таблица готова, дальше сравниваем группы приборов по типу, возрасту, станции и пакетам


In [30]:
# 8.1 Доля счётчиков с отрицательным / нереальным расходом по типу и возрасту
g81 = mm.groupby(['старый', 'tip']).agg(счётчиков=('bs', 'size'),
                                        отрицательных=('отрицательный', 'sum'),
                                        нереальных=('нереальный', 'sum'))
g81['доля отриц.'] = (g81['отрицательных'] / g81['счётчиков']).round(3)
g81['доля нереальных'] = (g81['нереальных'] / g81['счётчиков']).round(3)
display(g81)

grp = mm['старый'] & (mm['tip'] == 'AQUA2')
rest = ~grp
for col in ['отрицательный', 'нереальный']:
    k_grp, k_rest, k_all = mm.loc[grp, col].sum(), mm.loc[rest, col].sum(), mm[col].sum()
    print(f'{col}: старые AQUA2 {frac(k_grp, grp.sum())}; остальные {frac(k_rest, rest.sum())}; '
          f'доля группы среди всех таких счётчиков {frac(k_grp, k_all)}; '
          f'p Фишера = {fisher_exact(pd.crosstab(grp, mm[col]))[1]:.2g}')
print(f'ВЫВОД: старые AQUA2 — {frac(grp.sum(), len(mm))} парка — главный источник отрицательных и нереальных '
      f'значений; заменить или перепрошить их в первую очередь, при поиске реальных нарушителей учитывать их сбои')

счётчиков  отрицательных  нереальных  доля отриц.  \
старый tip                                                                     
False  AQUA2                      62              7           5        0.113   
       aqua2_nvt_wa_stm         1595             10           0        0.006   
       aqua2_stm                1011             23           7        0.023   
       aqua2set_fudan_stm          3              0           0        0.000   
True   AQUA2                     329             62          54        0.188   

                           доля нереальных  
старый tip                                  
False  AQUA2                         0.081  
       aqua2_nvt_wa_stm              0.000  
       aqua2_stm                     0.007  
       aqua2set_fudan_stm            0.000  
True   AQUA2                         0.164

отрицательный: старые AQUA2 62/329 = 18.8%; остальные 40/2671 = 1.5%; доля группы среди всех таких счётчиков 62/102 = 60.8%; p Фишера = 1.1e-35
нереальный: старые AQUA2 54/329 = 16.4%; остальные 12/2671 = 0.4%; доля группы среди всех таких счётчиков 54/66 = 81.8%; p Фишера = 3.6e-42
ВЫВОД: старые AQUA2 — 329/3000 = 11.0% парка — главный источник отрицательных и нереальных значений; заменить или перепрошить их в первую очередь, при поиске реальных нарушителей учитывать их сбои


In [31]:
# 8.2 Нулевое потребление по базовым станциям
g82 = mm.groupby('bs').agg(счётчиков=('tip', 'size'),
                           ноль_весь_период=('ноль_весь_период', 'sum'),
                           сумма_долей_нулей=('доля_нулей', 'sum'),
                           сумма_расходов=('ср_расход', 'sum'))
g82['доля нулевых счётчиков'] = (g82['ноль_весь_период'] / g82['счётчиков']).round(3)
g82['ср_доля_нулей'] = (g82['сумма_долей_нулей'] / g82['счётчиков']).round(3)
g82['ср_расход'] = (g82['сумма_расходов'] / g82['счётчиков']).round(3)
display(g82.drop(columns=['сумма_долей_нулей', 'сумма_расходов']).sort_values('доля нулевых счётчиков', ascending=False))

worst = g82['доля нулевых счётчиков'].idxmax()
w = mm['bs'] == worst
others = ~w
p82 = fisher_exact(pd.crosstab(w, mm['ноль_весь_период']))[1]
print(f'{worst}: ноль весь период {frac(mm.loc[w, "ноль_весь_период"].sum(), w.sum())}; '
      f'остальные станции {frac(mm.loc[others, "ноль_весь_период"].sum(), others.sum())}; p Фишера = {p82:.2g}')
print(f'средний расход: {worst} {mm.loc[w, "ср_расход"].sum():.2f}/{w.sum()} = {mm.loc[w, "ср_расход"].mean():.3f} м³/сут; '
      f'остальные {mm.loc[others, "ср_расход"].sum():.2f}/{others.sum()} = {mm.loc[others, "ср_расход"].mean():.3f} м³/сут')
print(f'ВЫВОД: на {worst} нулевых счётчиков в разы больше, чем на других станциях, и расход почти нулевой. '
      f'Это похоже на проблему станции или канала, поэтому проверять надо станцию {worst} целиком; '
      f'её нулевые счётчики в список нарушителей не включаем')

,счётчиков,ноль_весь_период,доля нулевых счётчиков,ср_доля_нулей,ср_расход
bs,,,,,
BS-04,300,62,0.207,0.978,0.004
BS-02,300,37,0.123,0.462,0.048
BS-10,300,31,0.103,0.435,0.251
BS-05,300,24,0.080,0.530,0.093
BS-08,300,24,0.080,0.513,0.106
BS-03,300,18,0.060,0.333,0.119
BS-01,300,7,0.023,0.390,0.080
BS-09,300,4,0.013,0.277,0.132
BS-07,300,2,0.007,0.200,0.116


BS-04: ноль весь период 62/300 = 20.7%; остальные станции 148/2700 = 5.5%; p Фишера = 9.5e-17
средний расход: BS-04 1.06/300 = 0.004 м³/сут; остальные 321.07/2700 = 0.119 м³/сут
ВЫВОД: на BS-04 нулевых счётчиков в разы больше, чем на других станциях, и расход почти нулевой. Это похоже на проблему станции или канала, поэтому проверять надо станцию BS-04 целиком; её нулевые счётчики в список нарушителей не включаем


In [32]:
# 8.3 Среднее число пакетов на счётчик и нулевой расход
one = (df_sorted['paketov_za_sutki'] == 1).sum()
print('строк с 1 пакетом:', frac(one, len(df_sorted)),
      '| строк с 24–25 пакетами:', df_sorted['paketov_za_sutki'].isin([24, 25]).sum())
mm['nvt'] = mm['tip'] == 'aqua2_nvt_wa_stm'
mm['группа_пакетов'] = pd.cut(mm['пакетов'], [0, 1.5, 3, 12, np.inf],
                              labels=['<= 1.5', '1.5–3', '3–12', '> 12'])
g83 = mm.groupby(['nvt', 'группа_пакетов'], observed=True).agg(
    счётчиков=('tip', 'size'),
    ноль_весь_период=('ноль_весь_период', 'sum'),
    сумма_долей_нулей=('доля_нулей', 'sum'))
g83['доля нулевых счётчиков'] = (g83['ноль_весь_период'] / g83['счётчиков']).round(3)
g83['ср_доля_нулей'] = (g83['сумма_долей_нулей'] / g83['счётчиков']).round(3)
display(g83.drop(columns='сумма_долей_нулей'))

low = mm['группа_пакетов'] == '<= 1.5'
high = mm['группа_пакетов'] == '> 12'
normal_pk = mm['группа_пакетов'] == '3–12'
for name, sel in [('aqua2_nvt_wa_stm', mm['nvt']), ('остальные типы', ~mm['nvt'])]:
    parts = []
    for label, grp_pk in [('3–12', normal_pk), ('<= 1.5', low), ('> 12', high)]:
        g = sel & grp_pk
        parts.append(f'при {label} пакетах {frac(mm.loc[g, "ноль_весь_период"].sum(), g.sum())}')
    print(f'{name}: ноль весь период ' + '; '.join(parts))
suspect = mm['ноль_весь_период'] & (low | (high & mm['nvt']))
print(f'ВЫВОД: счётчики со средним числом пакетов <= 1.5 молчат в разы чаще в обоих типах, при > 12 — у '
      f'aqua2_nvt_wa_stm (у остальных типов таких счётчиков {(~mm["nvt"] & high).sum()}). Число пакетов — '
      f'ранний признак неисправности: сначала проверять {int(suspect.sum())} счётчиков с таким числом пакетов '
      f'и нулём весь период')

строк с 1 пакетом: 443164/693443 = 63.9% | строк с 24–25 пакетами: 56085


счётчиков  ноль_весь_период  доля нулевых счётчиков  \
nvt   группа_пакетов                                                        
False <= 1.5                 95                16                   0.168   
      1.5–3                 171                 3                   0.018   
      3–12                 1119                21                   0.019   
      > 12                   20                 0                   0.000   
True  <= 1.5                202                45                   0.223   
      1.5–3                 147                10                   0.068   
      3–12                  829                21                   0.025   
      > 12                  417                94                   0.225   

                      ср_доля_нулей  
nvt   группа_пакетов                 
False <= 1.5                  0.699  
      1.5–3                   0.358  
      3–12                    0.134  
      > 12                    0.058  
True  <= 1.5                  0.874  
      1.5–3                   0.730  
      3–12                    0.534  
      > 12                    0.670

aqua2_nvt_wa_stm: ноль весь период при 3–12 пакетах 21/829 = 2.5%; при <= 1.5 пакетах 45/202 = 22.3%; при > 12 пакетах 94/417 = 22.5%
остальные типы: ноль весь период при 3–12 пакетах 21/1119 = 1.9%; при <= 1.5 пакетах 16/95 = 16.8%; при > 12 пакетах 0/20 = 0.0%
ВЫВОД: счётчики со средним числом пакетов <= 1.5 молчат в разы чаще в обоих типах, при > 12 — у aqua2_nvt_wa_stm (у остальных типов таких счётчиков 20). Число пакетов — ранний признак неисправности: сначала проверять 155 счётчиков с таким числом пакетов и нулём весь период


## Находки и вывод по пункту 8